# ISE Music Genre — nhật ký xây dựng lời giải cuối

Bài toán là dự đoán thể loại của một track Spotify từ 15 feature âm thanh. Notebook đi theo đúng thứ tự tôi giải bài: chuẩn bị môi trường, mở và kiểm tra dữ liệu, dựng validation chống leakage, tạo feature, huấn luyện từng model, ghép ensemble, calibration, rồi mới xuất submission.

Mỗi ô code bên dưới chỉ có một nhiệm vụ. Ô Markdown ngay trước nó ghi rõ câu hỏi đang cần trả lời, lý do thực hiện và kết quả cần kiểm tra.


## Chặng 0 — Chuẩn bị nơi làm việc

### Xác định môi trường chạy

Notebook phải chạy được cả trên Google Colab lẫn local. Ở bước này tôi chỉ cần biết mình có đang ở Colab hay không; nếu có, cell cài đúng major version của XGBoost và LightGBM để API training không thay đổi. Các tiện ích xử lý đường dẫn và CPU chưa cần nên chưa được import.


In [ ]:
import sys

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

if IN_COLAB:
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "xgboost>=3.4,<4", "lightgbm>=4.7,<5",
    ])

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Environment:", "Google Colab" if IN_COLAB else "Local")


### Tìm đủ bốn file đầu vào

Sau khi môi trường đã rõ, tôi mới xác định thư mục dữ liệu. Trên Colab, cell mở hộp thoại upload nếu còn thiếu file; ở local, nó thử thư mục hiện tại rồi thư mục bài thi. Nhiệm vụ duy nhất ở đây là trả về một `PROJECT_DIR` hợp lệ chứa đủ bốn CSV.


In [ ]:
from pathlib import Path

REQUIRED_FILES = (
    "train.csv",
    "test.csv",
    "sample_submission.csv",
    "genre_mapping.csv",
)

if IN_COLAB:
    PROJECT_DIR = Path("/content")
    missing = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
    if missing:
        print("Chọn và upload 4 file:", REQUIRED_FILES)
        colab_files.upload()
else:
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "train.csv").exists():
        PROJECT_DIR = PROJECT_DIR / "ISE_TRAINNING_TEST_23-8-2026"

PROJECT_DIR = PROJECT_DIR.resolve()
missing = [name for name in REQUIRED_FILES if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Thiếu {missing} trong {PROJECT_DIR}")

print("Data folder:", PROJECT_DIR)


## Chặng 1 — Mở dữ liệu, chưa vội nghĩ đến model

### Chuẩn bị công cụ khảo sát và các quy ước chung

Ở thời điểm này tôi chỉ cần NumPy, Pandas và `display`; chưa có lý do để import thư viện modeling. Cell cũng đặt tên cột, số lớp và seed để mọi bước sau dùng cùng một quy ước.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 20_260_823
MODEL_SEED = 42
N_SPLITS = 3
N_CLASSES = 112
TARGET = "track_genre"
ID_COLUMN = "track_id"
EXPECTED_LABELS = np.arange(N_CLASSES)


### Đọc dữ liệu gốc

Cell này chỉ nạp bốn CSV. Tôi giữ riêng `sample_submission` và `genre_mapping` vì chúng sẽ lần lượt làm chuẩn cho định dạng đầu ra và miền nhãn hợp lệ.


In [ ]:
train = pd.read_csv(PROJECT_DIR / "train.csv")
test = pd.read_csv(PROJECT_DIR / "test.csv")
sample_submission = pd.read_csv(PROJECT_DIR / "sample_submission.csv")
genre_mapping = pd.read_csv(PROJECT_DIR / "genre_mapping.csv")


### Nhìn kích thước và độ lệch lớp

Trước khi viết assertion, tôi cần biết mình đang cầm dữ liệu gì. Bảng đầu so sánh số dòng, số cột và số ID duy nhất của train/test; bảng sau tóm tắt số mẫu trên 112 genre để thấy ngay bài toán mất cân bằng lớp.


In [ ]:
display(pd.DataFrame({
    "data": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "unique_track_id": [train[ID_COLUMN].nunique(), test[ID_COLUMN].nunique()],
}))
display(train[TARGET].value_counts().describe().to_frame("class_count").T)


### Khóa các giả định về dữ liệu

Quan sát bằng mắt chưa đủ. Cell này biến các giả định quan trọng thành assertion: đúng 15 feature, đúng thứ tự schema, ID không trùng hoặc giao nhau, không có NaN/inf và miền nhãn đúng 0–111. Nếu một điều sai, pipeline dừng trước khi tạo validation hay model.


In [ ]:
FEATURES = [column for column in test.columns if column != ID_COLUMN]
y = train[TARGET].to_numpy(dtype=np.int64)

assert len(FEATURES) == 15
assert list(train.columns) == [ID_COLUMN, *FEATURES, TARGET]
assert list(test.columns) == [ID_COLUMN, *FEATURES]
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique
assert set(train[ID_COLUMN]).isdisjoint(test[ID_COLUMN])
assert not train[FEATURES].isna().any().any()
assert not test[FEATURES].isna().any().any()
assert np.isfinite(train[FEATURES].to_numpy(dtype=float)).all()
assert np.isfinite(test[FEATURES].to_numpy(dtype=float)).all()
assert np.array_equal(np.unique(y), EXPECTED_LABELS)
assert np.array_equal(
    np.sort(genre_mapping["genre_id"].unique()),
    EXPECTED_LABELS,
)

print("PASS — schema, ID, NaN/inf và 112 labels hợp lệ.")


## Chặng 2 — Dựng validation không tự lừa mình

### Tìm những track có bộ feature giống hệt nhau

Train được xếp theo các block target nên không thể chia theo vị trí. Nguy hiểm hơn, nhiều dòng có cùng toàn bộ 15 feature; nếu chúng rơi sang hai phía của một fold, model có thể ghi nhớ. Cell này chỉ tạo `group_id` từ feature và xác nhận mỗi group không chứa nhiều nhãn khác nhau.


In [ ]:
raw_train = train[FEATURES]
raw_test = test[FEATURES]
group_id = pd.util.hash_pandas_object(raw_train, index=False).astype("uint64")

mixed_label_groups = (
    pd.DataFrame({"group": group_id, "target": y})
    .groupby("group")["target"]
    .nunique()
    .gt(1)
    .sum()
)
assert mixed_label_groups == 0

print("Unique feature rows:", group_id.nunique())
print("Duplicate rows beyond first:", len(train) - group_id.nunique())


### Tạo ba fold stratified theo group

Bây giờ mới xuất hiện `StratifiedGroupKFold`: stratified để 112 genre được phân bố cân đối, group để mọi bản sao ở cùng một phía. Test không tham gia tạo fold.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
splits = list(splitter.split(raw_train, y, groups=group_id))


### Kiểm toán các fold vừa tạo

Một splitter đúng tên chưa bảo đảm kết quả đúng. Cell này kiểm tra mỗi dòng validation được gán đúng một fold, mỗi fold có đủ 112 lớp và không có group nào xuất hiện ở cả train lẫn validation.


In [ ]:
fold_assignment = np.full(len(train), -1, dtype=np.int8)
fold_audit = []

for fold, (train_idx, valid_idx) in enumerate(splits):
    fold_assignment[valid_idx] = fold
    train_groups = set(group_id.iloc[train_idx])
    valid_groups = set(group_id.iloc[valid_idx])
    fold_audit.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "valid_classes": np.unique(y[valid_idx]).size,
        "group_overlap": len(train_groups & valid_groups),
    })

assert np.all(fold_assignment >= 0)
assert all(row["valid_classes"] == N_CLASSES for row in fold_audit)
assert all(row["group_overlap"] == 0 for row in fold_audit)
assert (
    pd.DataFrame({"group": group_id, "fold": fold_assignment})
    .groupby("group")["fold"]
    .nunique()
    .max()
    == 1
)

display(pd.DataFrame(fold_audit))
print("PASS — đủ 112 lớp/fold và không có duplicate-feature leakage.")


## Chặng 3 — Biểu diễn 15 con số cho từng model

### Tạo feature số cho XGBoost

`key=11` không thật sự lớn hơn `key=1`, nên tôi mã hóa key và time signature theo one-hot, thêm biểu diễn vòng tròn/circle of fifths và một số interaction có ý nghĩa âm học. Hàm chỉ biến đổi từng dòng, không học thống kê từ train hay test.


In [ ]:
def engineered_features(frame):
    result = frame[FEATURES].copy()
    key = result.pop("key").astype(int)
    time_signature = result.pop("time_signature").astype(int)

    for value in range(12):
        result[f"key_{value}"] = (key == value).astype(np.int8)
    for value in range(6):
        result[f"time_signature_{value}"] = (
            time_signature == value
        ).astype(np.int8)

    angle = 2 * np.pi * key / 12
    result["key_sin"] = np.sin(angle)
    result["key_cos"] = np.cos(angle)
    fifth_angle = 2 * np.pi * ((key * 7) % 12) / 12
    result["circle_of_fifths_sin"] = np.sin(fifth_angle)
    result["circle_of_fifths_cos"] = np.cos(fifth_angle)

    key_mode = key + 12 * result["mode"].astype(int)
    for value in range(24):
        result[f"key_mode_{value}"] = (key_mode == value).astype(np.int8)

    result["acoustic_low_energy"] = result["acousticness"] * (1 - result["energy"])
    result["energy_loudness"] = result["energy"] * (result["loudness"] + 60)
    result["dance_energy"] = result["danceability"] * result["energy"]
    result["dance_valence"] = result["danceability"] * result["valence"]
    result["speech_explicit"] = result["speechiness"] * (
        1 + result["explicit"].astype(float)
    )
    result["instrumental_acoustic"] = (
        result["instrumentalness"] * result["acousticness"]
    )
    result["tempo_dance"] = result["tempo"] * result["danceability"]
    result["audio_missing"] = (
        result[["danceability", "speechiness", "valence", "tempo"]]
        .eq(0)
        .all(axis=1)
        .astype(np.int8)
    )
    return result.astype(np.float32)


xgb_train = engineered_features(train)
xgb_test = engineered_features(test)
assert list(xgb_train.columns) == list(xgb_test.columns)

print("XGBoost feature shape:", xgb_train.shape)


### Đánh dấu categorical feature cho LightGBM

LightGBM có thể xử lý trực tiếp bốn biến rời rạc. Cell này chỉ tạo hai bảng dành cho LightGBM và ép train/test dùng cùng domain category; Extra Trees vẫn dùng nguyên 15 feature thô.


In [ ]:
CATEGORICAL = ["explicit", "key", "mode", "time_signature"]
CATEGORY_DOMAINS = {
    "explicit": [False, True],
    "key": list(range(12)),
    "mode": [0, 1],
    "time_signature": list(range(6)),
}

lgb_train = raw_train.copy()
lgb_test = raw_test.copy()
for column in CATEGORICAL:
    fixed_dtype = pd.CategoricalDtype(categories=CATEGORY_DOMAINS[column])
    lgb_train[column] = lgb_train[column].astype(fixed_dtype)
    lgb_test[column] = lgb_test[column].astype(fixed_dtype)

assert list(lgb_train.columns) == list(lgb_test.columns)
print("LightGBM feature shape:", lgb_train.shape)


## Chặng 4 — Ghi lại các thử nghiệm sàng lọc

Tôi dùng cùng fold 0 để sàng lọc nhanh trước khi trả chi phí cho full cross-validation. Extra Trees cho một baseline đa dạng; XGBoost mạnh nhất; LightGBM không thắng riêng lẻ nhưng có lỗi khác XGBoost. Class weight làm điểm giảm, còn CatBoost CPU quá chậm cho notebook cần chạy lại.

| Thử nghiệm | Macro F1 screening | Quyết định |
|---|---:|---|
| Extra Trees, leaf=1 | 0.36413 | bỏ cấu hình |
| Extra Trees, leaf=2 | 0.36434 | giữ vì ổn định/diverse |
| Extra Trees, balanced | 0.36173 | bỏ |
| XGBoost raw | 0.39328 | baseline mạnh |
| XGBoost raw + prior | 0.39864 | giữ ý tưởng prior |
| XGBoost engineered + prior | 0.40003 | giữ |
| XGBoost class-weight nhẹ | 0.39834 | bỏ |
| LightGBM categorical + prior | 0.39386 | giữ vì diversity |
| CatBoost CPU | ước tính >3 giờ/lượt | bỏ vì không phù hợp Colab |

Từ nhật ký này, tôi khóa ba model cho vòng chạy đầy đủ. Các lựa chọn ensemble và calibration vẫn phải được kiểm tra bằng OOF ở những chặng sau.


## Chặng 5 — Huấn luyện ba góc nhìn độc lập

### Chuẩn bị model và cách ghi điểm

Đây là lúc các thư viện modeling mới xuất hiện. Cell chỉ định nghĩa metric Macro F1 và hàm ghi lại F1, accuracy, log loss, thời gian của mỗi fold; nó chưa huấn luyện model nào.


In [ ]:
import gc
import os
import time
import warnings

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, recall_score
from xgboost import XGBClassifier

N_JOBS = max(1, min(8, os.cpu_count() or 2))


def macro_f1(y_true, y_pred):
    return float(f1_score(
        y_true,
        y_pred,
        labels=EXPECTED_LABELS,
        average="macro",
        zero_division=0,
    ))


def save_metrics(rows, name, fold, valid_idx, probability, elapsed):
    prediction = probability.argmax(axis=1)
    row = {
        "model": name,
        "fold": fold,
        "macro_f1": macro_f1(y[valid_idx], prediction),
        "accuracy": accuracy_score(y[valid_idx], prediction),
        "log_loss": log_loss(
            y[valid_idx],
            probability,
            labels=EXPECTED_LABELS,
        ),
        "seconds": elapsed,
    }
    rows.append(row)
    print(
        f"fold={fold + 1} F1={row['macro_f1']:.6f} "
        f"acc={row['accuracy']:.6f} time={elapsed:.1f}s",
        flush=True,
    )


### Dành chỗ cho dự đoán OOF và test

Mỗi dòng train phải nhận xác suất từ đúng model chưa thấy fold của nó; đó là OOF. Với test, mỗi model cho ba dự đoán rồi lấy trung bình. Cell này chỉ khởi tạo các ma trận để ba cell training sau ghi kết quả vào.


In [ ]:
MODEL_NAMES = ("extra_trees", "lightgbm", "xgboost")
oof_probability = {
    name: np.zeros((len(train), N_CLASSES), dtype=np.float32)
    for name in MODEL_NAMES
}
test_probability = {
    name: np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for name in MODEL_NAMES
}
metric_rows = []
best_iterations = {"lightgbm": [], "xgboost": []}


### Model 1 — Extra Trees trên feature thô

Extra Trees yếu hơn boosting nhưng tạo một góc nhìn khác nhờ nhiều cây ngẫu nhiên. Cell này chỉ chạy Extra Trees qua ba fold, điền xác suất OOF/test và ghi metric của model này.


In [ ]:
raw_train_float = raw_train.astype(np.float32)
raw_test_float = raw_test.astype(np.float32)

for fold, (train_idx, valid_idx) in enumerate(splits):
    started = time.perf_counter()
    model = ExtraTreesClassifier(
        n_estimators=120,
        max_features=1.0,
        min_samples_leaf=2,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
    )
    model.fit(raw_train_float.iloc[train_idx], y[train_idx])
    valid_probability = model.predict_proba(raw_train_float.iloc[valid_idx])

    oof_probability["extra_trees"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["extra_trees"] += (
        model.predict_proba(raw_test_float) / N_SPLITS
    )
    save_metrics(
        metric_rows,
        "extra_trees",
        fold,
        valid_idx,
        valid_probability,
        time.perf_counter() - started,
    )
    del model, valid_probability
    gc.collect()


### Model 2 — LightGBM với categorical feature

LightGBM nhận bốn cột categorical đã chuẩn bị ở trên và dùng early stopping trên validation fold. Cell này chỉ tạo xác suất OOF/test của LightGBM và lưu số vòng tốt nhất ở từng fold.


In [ ]:
for fold, (train_idx, valid_idx) in enumerate(splits):
    started = time.perf_counter()
    model = LGBMClassifier(
        objective="multiclass",
        num_class=N_CLASSES,
        n_estimators=1_200,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        max_bin=255,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=3.0,
        reg_alpha=0.05,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
        verbosity=-1,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(
            lgb_train.iloc[train_idx],
            y[train_idx],
            eval_set=[(lgb_train.iloc[valid_idx], y[valid_idx])],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(60, verbose=False)],
            categorical_feature=CATEGORICAL,
        )
    valid_probability = model.predict_proba(lgb_train.iloc[valid_idx])

    oof_probability["lightgbm"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["lightgbm"] += model.predict_proba(lgb_test) / N_SPLITS
    best_iterations["lightgbm"].append(int(model.best_iteration_))
    save_metrics(
        metric_rows,
        "lightgbm",
        fold,
        valid_idx,
        valid_probability,
        time.perf_counter() - started,
    )
    del model, valid_probability
    gc.collect()


### Model 3 — XGBoost trên feature đã engineering

XGBoost là model đơn mạnh nhất trong screening. Cell này chỉ huấn luyện nó qua ba fold với early stopping, sau đó điền xác suất OOF/test và số vòng tốt nhất.


In [ ]:
for fold, (train_idx, valid_idx) in enumerate(splits):
    started = time.perf_counter()
    model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.06,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=5.0,
        reg_alpha=0.05,
        objective="multi:softprob",
        num_class=N_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        max_bin=256,
        n_jobs=N_JOBS,
        random_state=MODEL_SEED + fold,
        early_stopping_rounds=45,
    )
    model.fit(
        xgb_train.iloc[train_idx],
        y[train_idx],
        eval_set=[(xgb_train.iloc[valid_idx], y[valid_idx])],
        verbose=False,
    )
    valid_probability = model.predict_proba(xgb_train.iloc[valid_idx])

    oof_probability["xgboost"][valid_idx] = valid_probability.astype(np.float32)
    test_probability["xgboost"] += model.predict_proba(xgb_test) / N_SPLITS
    best_iterations["xgboost"].append(int(model.best_iteration + 1))
    save_metrics(
        metric_rows,
        "xgboost",
        fold,
        valid_idx,
        valid_probability,
        time.perf_counter() - started,
    )
    del model, valid_probability
    gc.collect()


### So sánh và kiểm tra đầu ra ba model

Trước khi ensemble, mọi ma trận xác suất phải hữu hạn và mỗi hàng phải có tổng gần 1. Cell này chỉ kiểm tra điều đó rồi đặt metric ba model cạnh nhau; chưa trộn hay hiệu chỉnh xác suất.


In [ ]:
for probabilities in [*oof_probability.values(), *test_probability.values()]:
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=2e-4)

base_metrics = pd.DataFrame(metric_rows)
display(base_metrics)
display(base_metrics.groupby("model")["macro_f1"].agg(["mean", "std"]))
print("Best iterations:", best_iterations)
print(f"Total model runtime: {base_metrics['seconds'].sum():.1f}s")


## Chặng 6 — Ba xác suất trở thành một quyết định

### Trộn ba model

XGBoost mạnh nhất nên nhận trọng số 0.40; Extra Trees nhận 0.35 vì diversity; LightGBM nhận 0.25. Cell này chỉ lấy trung bình có trọng số của ba ma trận xác suất, chưa đụng đến class prior hay threshold.


In [ ]:
WEIGHTS = {
    "xgboost": 0.40,
    "lightgbm": 0.25,
    "extra_trees": 0.35,
}
assert np.isclose(sum(WEIGHTS.values()), 1.0)

oof_blend = sum(
    WEIGHTS[name] * oof_probability[name]
    for name in WEIGHTS
)
test_blend = sum(
    WEIGHTS[name] * test_probability[name]
    for name in WEIGHTS
)


### Điều chỉnh OOF theo class prior của từng training fold

Macro F1 cho mỗi genre tiếng nói ngang nhau dù số mẫu rất lệch. Tôi dùng prior correction nhẹ với alpha 0.35. Để tránh leakage, prior của mỗi validation row chỉ được tính từ phần training của fold tương ứng. Cell này chỉ tạo `oof_adjusted`.


In [ ]:
PRIOR_ALPHA = 0.35
oof_adjusted = np.empty_like(oof_blend, dtype=np.float64)

for train_idx, valid_idx in splits:
    fold_count = np.bincount(y[train_idx], minlength=N_CLASSES).astype(float)
    fold_prior = fold_count / fold_count.sum()
    adjusted = oof_blend[valid_idx] / np.power(
        fold_prior[None, :],
        PRIOR_ALPHA,
    )
    oof_adjusted[valid_idx] = adjusted / adjusted.sum(axis=1, keepdims=True)

assert np.isfinite(oof_adjusted).all()
assert np.allclose(oof_adjusted.sum(axis=1), 1)


### Chấm ensemble trước calibration

Cell này chuyển xác suất OOF đã điều chỉnh thành nhãn và báo Macro F1 theo từng fold lẫn toàn bộ OOF. Đây là mốc đối chứng để biết threshold calibration ở chặng sau có thật sự cải thiện hay không.


In [ ]:
base_oof_prediction = oof_adjusted.argmax(axis=1)
fold_scores = [
    macro_f1(y[valid_idx], base_oof_prediction[valid_idx])
    for _, valid_idx in splits
]
ensemble_summary = pd.DataFrame({
    "fold": range(N_SPLITS),
    "macro_f1": fold_scores,
})

display(ensemble_summary)
print(f"Fold mean: {np.mean(fold_scores):.6f}")
print(f"Fold std:  {np.std(fold_scores, ddof=1):.6f}")
print(f"Global OOF Macro F1: {macro_f1(y, base_oof_prediction):.6f}")
print(f"Global OOF accuracy: {accuracy_score(y, base_oof_prediction):.6f}")
print(
    "Global OOF macro recall:",
    f"{recall_score(y, base_oof_prediction, labels=EXPECTED_LABELS, average='macro', zero_division=0):.6f}",
)


## Chặng 7 — Không để genre nhỏ biến mất

### Định nghĩa cách học threshold cho từng lớp

Từ diagnostic OOF, vài genre hiếm vẫn bị lớp đông lấn át. Cell này chỉ định nghĩa hàm tìm threshold tối đa hóa F1 nhị phân trên precision–recall curve cho từng genre; nó chưa chọn gamma và chưa nhìn test.


In [ ]:
from sklearn.metrics import precision_recall_curve


def fit_class_thresholds(mask):
    thresholds = np.empty(N_CLASSES, dtype=np.float64)
    for class_id in EXPECTED_LABELS:
        precision, recall, candidates = precision_recall_curve(
            y[mask] == class_id,
            oof_adjusted[mask, class_id],
        )
        binary_f1 = 2 * precision * recall / np.maximum(
            precision + recall,
            1e-15,
        )
        best_index = int(np.nanargmax(binary_f1[:-1]))
        thresholds[class_id] = max(float(candidates[best_index]), 1e-8)
    return thresholds


### Chọn mức ảnh hưởng của threshold bằng cross-fold

Đây là chỗ dễ tự lừa mình nhất. Với mỗi held-out fold, cell học 112 threshold từ hai fold còn lại rồi chấm nhiều giá trị gamma trên fold chưa tham gia học. Gamma được chọn theo mean ba held-out scores, không phải theo điểm fit trên cùng dữ liệu.


In [ ]:
gamma_grid = np.round(np.arange(0.0, 1.01, 0.1), 1)
crossfold_scores = {gamma: [] for gamma in gamma_grid}

for held_out_fold in range(N_SPLITS):
    calibration_mask = fold_assignment != held_out_fold
    validation_mask = fold_assignment == held_out_fold
    thresholds = fit_class_thresholds(calibration_mask)

    for gamma in gamma_grid:
        prediction = (
            oof_adjusted[validation_mask]
            / np.power(thresholds[None, :], gamma)
        ).argmax(axis=1)
        crossfold_scores[gamma].append(
            macro_f1(y[validation_mask], prediction)
        )

calibration_report = pd.DataFrame([
    {
        "gamma": gamma,
        "fold_0": scores[0],
        "fold_1": scores[1],
        "fold_2": scores[2],
        "mean": np.mean(scores),
        "std": np.std(scores, ddof=1),
    }
    for gamma, scores in crossfold_scores.items()
])
BEST_GAMMA = float(
    calibration_report.loc[calibration_report["mean"].idxmax(), "gamma"]
)

display(calibration_report)
print(f"Selected gamma: {BEST_GAMMA}")
print(f"Cross-fold calibrated mean: {calibration_report['mean'].max():.6f}")


### Fit calibration cuối và dự đoán test

Sau khi gamma đã được kiểm chứng cross-fold, tôi mới fit threshold trên toàn bộ OOF. Test prior chỉ lấy từ toàn bộ train; phân phối dự đoán test không được dùng để chỉnh luật quyết định. Cell này chỉ tạo `test_prediction`.


In [ ]:
final_thresholds = fit_class_thresholds(
    np.ones(len(train), dtype=bool)
)
full_count = np.bincount(y, minlength=N_CLASSES).astype(float)
full_prior = full_count / full_count.sum()
test_adjusted = test_blend / np.power(
    full_prior[None, :],
    PRIOR_ALPHA,
)
test_adjusted /= test_adjusted.sum(axis=1, keepdims=True)
test_prediction = (
    test_adjusted
    / np.power(final_thresholds[None, :], BEST_GAMMA)
).argmax(axis=1)

print("Predicted test classes:", np.unique(test_prediction).size, "/", N_CLASSES)


## Chặng cuối — Đóng phong bì submission

### Dựng và kiểm tra bảng submission trong bộ nhớ

Một model tốt vẫn có thể nhận điểm 0 nếu CSV sai format. Cell này chỉ tạo DataFrame và khóa đúng hai cột, đúng số dòng/thứ tự ID, ID duy nhất và nhãn nằm trong miền hợp lệ. Chưa có file nào được ghi trước khi toàn bộ assertion pass.


In [ ]:
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET: test_prediction.astype(np.int64),
})

assert list(submission.columns) == [ID_COLUMN, TARGET]
assert len(submission) == len(test) == len(sample_submission)
assert submission[ID_COLUMN].equals(test[ID_COLUMN])
assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
assert submission[ID_COLUMN].is_unique
assert submission[TARGET].between(0, N_CLASSES - 1).all()
assert set(submission[TARGET]).issubset(set(train[TARGET]))

display(submission.head())
print("PASS — submission đúng schema, thứ tự ID và miền nhãn.")


### Ghi file cuối

Chỉ sau khi bảng đã hợp lệ, cell cuối mới ghi CSV và báo lại các thống kê đủ để kiểm tra nhanh. Nếu chạy trên Colab, chính file vừa ghi sẽ được tải xuống.


In [ ]:
OUTPUT_PATH = PROJECT_DIR / "submission_final_macro_f1_ensemble.csv"
submission.to_csv(OUTPUT_PATH, index=False)

prediction_counts = submission[TARGET].value_counts().sort_index()
print("Saved:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("Unique IDs:", submission[ID_COLUMN].nunique())
print("Predicted classes:", submission[TARGET].nunique())
print(
    "Prediction count min/median/max:",
    int(prediction_counts.min()),
    float(prediction_counts.median()),
    int(prediction_counts.max()),
)

if IN_COLAB:
    colab_files.download(str(OUTPUT_PATH))


## Vì sao tôi bấm Submit

Tôi không bấm Submit chỉ vì thấy một con số lớn nhất. Tôi bấm vì chuỗi bằng chứng nối được với nhau:

1. Chiếc cân validation giữ class balance và chặn duplicate leakage.
2. XGBoost tự nó đã vượt mốc mục tiêu trên mean 3 fold.
3. LightGBM và Extra Trees chỉ được giữ vì diversity của chúng cải thiện cả ba fold.
4. Prior và threshold phục vụ đúng metric Macro F1; threshold còn được thử trên fold chưa tham gia học.
5. Test không quyết định feature, model, weight, prior hay threshold; nó chỉ được transform theo quy tắc cố định và predict.
6. CSV cuối đã qua mọi kiểm tra schema, ID và label rồi mới được ghi.

Ở lượt chạy local, model tự nhiên chọn 111/112 lớp trên test. Tôi đã cân nhắc ép một row cho lớp còn thiếu, nhưng coverage repair không hề kích hoạt trên ba held-out fold nên không có bằng chứng validation cho thao tác đó. Tôi giữ dự đoán nguyên bản thay vì sửa test chỉ để bảng phân phối trông đẹp; 111 lớp vẫn là submission hợp lệ.

Macro F1 leaderboard vẫn có thể khác OOF do test sampling hoặc distribution shift. Không ai có thể đảm bảo chính xác 0.400 khi test không có nhãn. Nhưng với held-out cross-fold quanh 0.417 và một pipeline chạy lại được từ đầu, đây là submission mà tôi đủ tự tin để ký tên.

## Mốc 0.399 → 0.402 — threshold calibration đã cứu submission như thế nào

Trước notebook hoàn chỉnh, tôi từng chạy trực tiếp script `build_submission_from_scratch.py`. Script này đã train đúng ensemble ba model và dùng weights XGBoost / LightGBM / Extra Trees = `0.40 / 0.25 / 0.35`, sau đó chia xác suất cho `class_prior ** 0.35` rồi argmax. Nó **chưa có threshold riêng cho từng lớp**. File đầu ra dùng chính tên `submission_final_macro_f1_ensemble.csv`; người dùng đem nộp và nhận **Public Macro F1 = 0.399**.

Log của lần chạy sơ bộ còn trong lịch sử hội thoại: file có 21,947 rows, dự đoán **111/112 lớp**, thiếu `genre_id=56` (`indie`), với số dự đoán mỗi lớp min/median/max là `4 / 219 / 413`. Sau đó notebook hoàn chỉnh ghi đè cùng đường dẫn nên byte của CSV 0.399 không còn trên filesystem; đây là lý do audit file đơn thuần đã bỏ sót mốc này.

Thay đổi quyết định giữa hai lần submit không phải kiến trúc model mà là **OOF per-class threshold calibration**:

1. Với mỗi genre, dùng OOF probability để tìm threshold tối ưu F1 trên precision–recall curve.
2. Học threshold trên các fold còn lại rồi chấm fold đang giữ, tránh báo điểm calibration trên chính dữ liệu đã học threshold.
3. Dùng `gamma=0.8` để làm mềm mức tác động của 112 threshold.
4. Fit threshold cuối trên toàn bộ OOF rồi mới áp dụng cho test.

Trước bước này, ensemble + prior adjustment có mean ba fold **0.413009**. Sau cross-fold threshold calibration, mean tăng lên **0.416984**, tức `+0.003975`. Public cũng tăng từ **0.399 lên 0.402**, gần đúng cùng cỡ `+0.003`. Đây là bằng chứng mạnh nhất rằng calibration—không phải một model mới—đã tạo ra cú nhảy đầu tiên.

Chi tiết '114/115 nhãn' trong trí nhớ cần hiệu chỉnh: dataset có đúng **112 lớp (0–111)**. Quan trọng hơn, file 0.402 vẫn chỉ dự đoán **111/112 lớp**, lần này thiếu lớp 53 (`house`); file đầu tiên dự đoán đủ 112/112 là lượt 1 nhưng Public vẫn 0.402. Vì vậy cú tăng 0.003 **không đến từ việc ép cho đủ nhãn**. Hai file đều thiếu một lớp; calibration đã phân phối lại decision boundary giữa 112 genre tốt hơn.

# Hậu truyện — năm lần nhích Public Leaderboard

Câu chuyện chưa kết thúc khi CSV đầu tiên ra đời. Submission gốc đạt **Public Macro F1 = 0.402** trên **51% test**. Đây là một kết quả đáng mừng vì nó xác nhận validation không hoàn toàn viển vông, nhưng khoảng cách giữa cross-fold khoảng 0.417 và Public 0.402 cũng cảnh báo rằng tôi chưa được phép tin tuyệt đối vào OOF.

Chúng tôi thống nhất làm thêm đúng năm vòng theo nhịp rất chậm: mỗi lượt chỉ thay đổi một yếu tố, giữ lại mọi CSV và chỉ nhìn Public sau khi đã chốt file. Mục tiêu không phải dò ngẫu nhiên cho đến khi bảng điểm may mắn tăng, mà là xem một cải tiến nhỏ có đi cùng chiều trên OOF và 51% test hay không.

Để giữ đúng dấu vết từng thời kỳ, cấu hình và code phía trên vẫn là snapshot của submission gốc 0.402; tôi không sửa ngược lịch sử sau khi đã biết leaderboard. Output cũ được xóa khi tách lại cell, vì vậy cần chạy tuần tự notebook để tái tạo các bảng kết quả. Phần dưới ghi lại các script/CSV kế tiếp và chốt cấu hình khuyến nghị mới.

> Public chỉ có 51% test và điểm hiển thị ba chữ số thập phân. Hai submission cùng 0.402 chưa chắc có điểm thật hoàn toàn bằng nhau; ngược lại, một bước tăng 0.001 cũng chưa bảo đảm sẽ tăng trên 49% Private.

## Nhật ký bảy mốc submission

| Mốc | Thay đổi duy nhất | Cross-fold Macro F1 | Dòng đổi so với mốc trước | Public F1 | Kết luận |
|---|---|---:|---:|---:|---|
| Sơ bộ | Ensemble 3-fold + prior, chưa có class threshold | 0.413009 (3-fold) | — | **0.399** | CSV từ script Python, sau đó bị ghi đè |
| Gốc | Thêm OOF per-class threshold, gamma 0.80 | 0.416984 (3-fold) | Không còn CSV cũ để đếm | **0.402** | Cú tăng Public đầu tiên |
| Lượt 1 | `n_splits: 3 → 5` | 0.425578 | 2,803 (12.77%) | **0.402** | OOF tăng nhưng Public đứng yên |
| Lượt 2 | `gamma: 0.80 → 0.75` | 0.426203 | 165 (0.75%) | **0.402** | Thay đổi quá nhỏ để thấy chuyển động |
| Lượt 3 | `prior alpha: 0.35 → 0.30` | 0.426462 | 347 (1.58%) | **0.403** | Cải tiến Public duy nhất |
| Lượt 4 | `prior alpha: 0.300 → 0.275` | 0.426591 | 325 (1.48%) | **0.403** | Chạm vùng plateau |
| Lượt 5 | Dịch 0.025 weight LightGBM → Extra Trees | **0.427082** | 623 (2.84%) | **0.402** | OOF cao nhất nhưng Public giảm |

Hai mốc sơ bộ/gốc dùng cùng validation 3-fold nên mức tăng 0.413009 → 0.416984 phản ánh trực tiếp threshold calibration. Chúng không so sánh tuyệt đối với năm mốc 5-fold. Từ lượt 1 đến lượt 5, fold, seed, feature và xác suất model được giữ cố định; vì vậy các chênh lệch nhỏ trong đoạn này có ý nghĩa so sánh trực tiếp hơn.

## Chúng tôi đã học được gì từ từng cú nhích?

### 1. Nhiều dữ liệu train hơn chưa chắc làm leaderboard nhúc nhích

Ở lượt 1, đổi từ 3-fold sang 5-fold khiến mỗi model học từ 80% thay vì 66.7% train. Cả ba base model đều tăng OOF: mean XGBoost từ khoảng 0.4040 lên 0.4146, LightGBM lên 0.4096 và Extra Trees lên 0.3758. Ensemble calibrated cũng lên 0.4256. Dù vậy Public vẫn là 0.402. Kết luận hợp lý không phải là 5-fold vô dụng, mà là phần tăng OOF chưa đủ lớn hoặc chưa đúng loại lỗi đang quyết định Public.

### 2. Gamma nằm trên một plateau rộng

Lượt 2 chỉ đổi threshold gamma từ 0.80 xuống 0.75. Cross-fold tăng 0.000625 nhưng chỉ 165 test rows đổi nhãn, nên Public vẫn làm tròn ở 0.402. Điều này xác nhận gamma 0.75–0.80 là một vùng khá phẳng; tiếp tục dò thêm chữ số ở đây dễ trở thành leaderboard overfitting.

### 3. Giảm nhẹ prior correction là thay đổi thật sự có ích

Lượt 3 giảm alpha từ 0.35 xuống 0.30. Alpha nhỏ hơn nghĩa là bớt khuếch đại xác suất của lớp hiếm một chút. Chỉ 347 rows đổi nhãn nhưng Public tăng lên 0.403. Đây là tín hiệu quan trọng: recipe cũ có vẻ đã bù class imbalance hơi mạnh. Khi giảm tiếp xuống 0.275 ở lượt 4, Public không tăng thêm; chúng tôi đã chạm plateau và dừng thay vì tiếp tục kéo alpha xuống.

### 4. Đỉnh OOF không phải đích đến

Lượt 5 là bài học đắt giá nhất. Dịch 0.025 trọng số từ LightGBM sang Extra Trees nâng mean cross-fold từ 0.426591 lên **0.427082**, cao nhất toàn bộ vòng lặp. Nhưng độ lệch chuẩn giữa fold cũng tăng từ 0.006689 lên 0.007791, và Public rơi về 0.402. Một mean tốt hơn 0.00049 không đủ bù cho sự thiếu ổn định. Nếu chỉ nhìn dòng OOF cao nhất, chúng tôi đã chọn sai submission.

## Quyết định cuối — quay về lượt 3

Submission tôi chọn để giữ làm ứng viên cuối là **`submission_iter3_prior_alpha_0p30.csv`** với cấu hình:

- 5-fold `StratifiedGroupKFold`;
- weights XGBoost / LightGBM / Extra Trees = `0.40 / 0.25 / 0.35`;
- prior alpha = `0.30`;
- threshold gamma = `0.75`;
- Public Macro F1 = **0.403** trên 51% test.

Lượt 4 cũng đạt 0.403 và OOF mean nhỉnh hơn 0.000129, nhưng lượt 3 có cross-fold standard deviation thấp hơn (`0.006087` so với `0.006689`), ít bước dò Public hơn và nằm ở một giá trị alpha tròn. Vì 49% Private vẫn chưa được nhìn thấy, đó là lựa chọn bảo thủ hợp lý hơn.

Các file qua từng thời kỳ đều được giữ nguyên khi có thể:

- bản sơ bộ 0.399 không còn byte riêng vì từng dùng cùng tên file và bị notebook ghi đè; cấu hình/log đã được phục dựng trong `artifacts/leaderboard_loop/preliminary_0399_reconstruction.json`;
- `submission_final_macro_f1_ensemble.csv` — mốc gốc 0.402;
- `submission_iter1_5fold_same_recipe.csv` — lượt 1, 0.402;
- `submission_iter2_gamma_0p75.csv` — lượt 2, 0.402;
- **`submission_iter3_prior_alpha_0p30.csv` — lượt 3, 0.403, khuyến nghị cuối;**
- `submission_iter4_prior_alpha_0p275.csv` — lượt 4, 0.403;
- `submission_iter5_weight_lgb0p225_et0p375.csv` — lượt 5, 0.402.

Điểm tăng chỉ 0.001, nhưng vòng lặp đã cho một kết luận lớn hơn con số ấy: **dùng OOF để đề xuất, dùng leaderboard thật tiết kiệm để kiểm tra, và ưu tiên vùng ổn định thay vì cực đại mong manh**. Đó là lý do chúng tôi dừng ở 0.403 thay vì tiếp tục mò cho đến khi vô tình overfit 51% Public.